<a href="https://colab.research.google.com/github/mofermino/condensed-subject-matters/blob/main/QHE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh  # for Hermitian eigensolver

# Parameters
N = 126              # number of sites along x (finite)
alpha = 1/3          # magnetic flux per plaquette (rational)
t = 1.0              # hopping parameter
ky = np.pi/2            # transverse momentum

# Construct Hamiltonian
def harper_hamiltonian(N, alpha, ky, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase)  # On-site cosine term
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Build and diagonalize
H = harper_hamiltonian(N, alpha, ky, t)
eigvals, eigvecs = eigh(H)

# Plot eigenvalue spectrum
plt.figure(figsize=(8, 8))
plt.plot(np.arange(len(eigvals)), eigvals, 'o')
plt.title('Harper Model Spectrum')
plt.xlabel('State Index')
plt.ylabel('Energy')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot some edge-localized wavefunctions
plt.figure(figsize=(8, 8))
for i in [0,1]:  # ground and highest energy states
    psi = (eigvecs[:, i])**2
    plt.plot(psi, label=f"State {i}")
plt.title('Wavefunction Localization')
plt.xlabel('Site (x direction)')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, inv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, inv
from tqdm import tqdm  # progress bar

# Parameters
N = 92               # number of sites along x (finite)
alpha = 1/2           # magnetic flux per plaquette (rational)
t = 1.0               # hopping parameter
ky_vals = np.linspace(-np.pi, np.pi, 200)  # sweep over ky
# Parameters for LDOS map
E_vals = np.linspace(-np.pi, np.pi, 200)  # energy sweep
ldos_map = np.zeros((N, len(E_vals)))  # store LDOS per site and energy
# Compute LDOS map
# Store eigenvalues for butterfly plot
spectrum = []

# Construct Harper Hamiltonian
def harper_hamiltonian(N, alpha, ky, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase)
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Calculate spectrum for ky sweep
for ky in ky_vals:
    H = harper_hamiltonian(N, alpha, ky, t)
    eigvals, _ = eigh(H)
    spectrum.append(eigvals)

# Convert to array for plotting
spectrum = np.array(spectrum)

# Plot Hofstadter-like spectrum vs ky
plt.figure(figsize=(10, 6))
for i in range(spectrum.shape[1]):
    plt.plot(ky_vals, spectrum[:, i], 'b-', linewidth=0.5)
plt.title('Harper Model Spectrum (Hofstadter-like)')
plt.xlabel(r'$k_y$')
plt.ylabel('Energy')
plt.grid(True)
plt.tight_layout()
plt.show()


# Green's function
def recursive_greens_function(E, N, alpha, ky, eta=1e-3):
    # Calculate the Green's function G = 1 / ((E + i*eta)*I - H)
    G = np.linalg.inv((E + 1j * eta) * np.eye(H.shape[0]) - H)
    # LDOS is proportional to -Im(G_ii)
    LDOS = -np.imag(np.diag(G)) / np.pi
    return LDOS


# Calculate LDOS for an example energy and ky
example_E = 0.5
example_ky = 0.0
H_example = harper_hamiltonian(N, alpha, example_ky, t)
ldos = recursive_greens_function(example_E, H_example)

# Note: The following loops and plots related to calculating and plotting a "spectrum" using the Green's function appear to be
# a misunderstanding of how Green's functions are typically used to calculate LDOS or transport.
# They are commented out as they don't align with standard Green's function applications for this model.

#for E in E_vals:
#     G = recursive_greens_function(N, alpha, ky, E_vals)
#     eigvals, _ = eigh(G)
#     spectrum.append(eigvals)

# spectrum = np.array(spectrum)

# Plot Hofstadter-like spectrum vs ky
# plt.figure(figsize=(10, 6))
# for i in range(spectrum.shape[0]):
#     plt.plot(E_vals, spectrum[:, i], 'b-', linewidth=0.5)
# plt.grid(False)
# plt.tight_layout()
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm  # progress bar

# Parameters for LDOS map
E_vals = np.linspace(-3, 3, 200)  # energy sweep
ldos_map = np.zeros((N, len(E_vals)))  # store LDOS per site and energy

# Use fixed ky for now; can later extend to 2D LDOS(E, ky)
fixed_ky = 0.5

# Compute LDOS map
for i, E in enumerate(tqdm(E_vals, desc="Computing LDOS map")):
    ldos_map[:, i] = recursive_greens_function(E, N, alpha, fixed_ky)

# Plot LDOS heatmap
plt.figure(figsize=(10, 6))
plt.imshow(ldos_map, aspect='auto', origin='lower', extent=[E_vals[0], E_vals[-1], 0, N],
           cmap='inferno', interpolation='gaussian')
plt.colorbar(label='LDOS')
plt.xlabel('Energy')
plt.ylabel('Site Index (x)')
plt.title('LDOS vs Energy and Position (ky = 0)')
plt.tight_layout()
plt.show()


In [ ]:
# Self-energy function for semi-infinite 1D chain lead
def surface_self_energy(t, E, eta=1e-3):
    z = E + 1j * eta
    return (z - np.sqrt(z**2 - 4 * t**2)) / 2

# Add self-energy correction to boundary sites (lead at m=0 and m=N-1)
def harper_with_self_energy(N, alpha, ky, E, t=1.0, eta=1e-3):
    H = harper_hamiltonian(N, alpha, ky, t)
    sigma = surface_self_energy(t, E, eta)
    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = inv((E + 1j * eta) * np.eye(N) - H)
    LDOS = -np.imag(np.diag(G)) / np.pi
    return LDOS

# Compute LDOS map with self-energy correction
ldos_self_map = np.zeros((N, len(E_vals)))

for i, E in enumerate(tqdm(E_vals, desc="LDOS with self-energy")):
    ldos_self_map[:, i] = harper_with_self_energy(N, alpha, fixed_ky, E, t)

# Plot updated LDOS map
plt.figure(figsize=(10, 6))
plt.imshow(ldos_self_map, aspect='auto', origin='lower', extent=[E_vals[0], E_vals[-1], 0, N],
           cmap='inferno', interpolation='none')
plt.colorbar(label='LDOS')
plt.xlabel('Energy')
plt.ylabel('Site Index (x)')
plt.title('LDOS with Self-Energy from Semi-Infinite Leads (ky = 0)')
plt.tight_layout()
plt.show()


In [ ]:
# Parameters for 2D LDOS map
ky_vals_2D = np.linspace(-np.pi, np.pi, 100)
E_vals_2D = np.linspace(-3, 3, 150)

# Only collect LDOS at a specific site (e.g., edge site m = 0)
site_index = 0
ldos_2D_map = np.zeros((len(ky_vals_2D), len(E_vals_2D)))

# Compute LDOS(E, ky) at the selected site
for i, ky in enumerate(tqdm(ky_vals_2D, desc="Computing 2D LDOS")):
    for j, E in enumerate(E_vals_2D):
        ldos_profile = harper_with_self_energy(N, alpha, ky, E, t)
        ldos_2D_map[i, j] = ldos_profile[site_index]

# Plot 2D LDOS map
plt.figure(figsize=(10, 6))
plt.imshow(ldos_2D_map, aspect='auto', origin='lower',
           extent=[E_vals_2D[0], E_vals_2D[-1], ky_vals_2D[0], ky_vals_2D[-1]],
           cmap='inferno', interpolation='bilinear')
plt.colorbar(label='LDOS at Site 0')
plt.xlabel('Energy')
plt.ylabel(r'$k_y$')
plt.title(r'2D LDOS($E$, $k_y$) at Edge Site $m=0$ with Self-Energy')
plt.tight_layout()
plt.show()


In [ ]:

# Plot 2D LDOS map
plt.figure(figsize=(10, 6))
plt.imshow(ldos_2D_map, aspect='auto', origin='lower',
           extent=[E_vals_2D[0], E_vals_2D[-1], ky_vals_2D[0], ky_vals_2D[-1]],
           cmap='inferno', interpolation='bilinear')
plt.colorbar(label='LDOS at Site 0')
plt.xlabel('Energy')
plt.ylabel(r'$k_y$')
plt.title(r'2D LDOS($E$, $k_y$) at Edge Site $m=0$ with Self-Energy')
plt.tight_layout()
plt.show()


In [ ]:
# Fix missing import by redefining function with correct numpy inversion
def transmission_fixed_v2(N, alpha, ky, E, t=1.0, eta=1e-3):
    H = harper_hamiltonian(N, alpha, ky, t)
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))

    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(N) - H)

    # Use scalar Caroli formula
    T = np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)
    return T

# Recompute transmission with fixed function
transmission_vals_fixed = []
E_transport = 0.5  # Energy at which transport is evaluated
for ky in tqdm(ky_vals_2D, desc="Computing corrected transmission"):
    T_val = transmission_fixed_v2(N, alpha, ky, E_transport)
    transmission_vals_fixed.append(T_val)

# Plot corrected transmission vs ky
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_2D, transmission_vals_fixed, 'r-')
plt.xlabel(r'$k_y$')
plt.ylabel('Transmission')
plt.title(f'Transmission vs $k_y$ at E = {E_transport}')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# 1. Generate 2D transmission map over (E, ky)
E_vals_trans = np.linspace(-np.pi, np.pi, 300)
T_2D_map = np.zeros((len(ky_vals_2D), len(E_vals_trans)))

# Compute transmission at all (E, ky)
for i, ky in enumerate(tqdm(ky_vals_2D, desc="Computing 2D Transmission Map")):
    for j, E in enumerate(E_vals_trans):
        T_2D_map[i, j] = transmission_fixed_v2(N, alpha, ky, E)

# Plot 2D transmission map
plt.figure(figsize=(10, 6))
plt.imshow(T_2D_map, aspect='auto', origin='lower',
           extent=[E_vals_trans[0], E_vals_trans[-1], ky_vals_2D[0], ky_vals_2D[-1]],
           cmap='plasma', interpolation='none')
plt.colorbar(label='Transmission')
plt.xlabel('Energy')
plt.ylabel(r'$k_y$')
plt.title('2D Transmission Map T(E, $k_y$)')
plt.tight_layout()
plt.show()


In [ ]:
# Function to add disorder to the Harper Hamiltonian
def harper_with_disorder(N, alpha, ky, disorder_strength=1.0, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        onsite = 2 * t * np.cos(phase)
        disorder = disorder_strength * (2 * np.random.rand() - 1)  # Uniform [-1, 1]
        H[m, m] = onsite + disorder
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Function to compute transmission with disorder
def transmission_disorder(N, alpha, ky, E, disorder_strength=1.0, t=1.0, eta=1e-3):
    H = harper_with_disorder(N, alpha, ky, disorder_strength, t)
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))

    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(N) - H)

    T = np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)
    return T

# Sweep over ky for a single realization of disorder
disorder_strength = 0.5  # Strong disorder
transmission_disorder_vals = []

for ky in tqdm(ky_vals_2D, desc="Transmission with Disorder"):
    T_dis = transmission_disorder(N, alpha, ky, E_transport, disorder_strength)
    transmission_disorder_vals.append(T_dis)

# Plot transmission with disorder
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_2D, transmission_disorder_vals, 'g-', label='Disordered')
plt.xlabel(r'$k_y$')
plt.ylabel('Transmission')
plt.title(f'Transmission vs $k_y$ at E = {E_transport} with Disorder (W = {disorder_strength})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Function to add disorder to the Harper Hamiltonian
def harper_with_disorder(N, alpha, ky, disorder_strength=1.0, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        onsite = 2 * t * np.cos(phase)
        disorder = disorder_strength * (2 * np.random.rand() - 1)  # Uniform [-1, 1]
        H[m, m] = onsite + disorder
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Function to compute transmission with disorder
def transmission_disorder(N, alpha, ky, E, disorder_strength=1.0, t=1.0, eta=1e-3):
    H = harper_with_disorder(N, alpha, ky, disorder_strength, t)
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))

    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(N) - H)

    T = np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)
    return T

# Sweep over ky for a single realization of disorder
disorder_strength = 1.0  # Strong disorder
transmission_disorder_vals = []

for ky in tqdm(ky_vals_2D, desc="Transmission with Disorder"):
    T_dis = transmission_disorder(N, alpha, ky, E_transport, disorder_strength)
    transmission_disorder_vals.append(T_dis)

# Plot transmission with disorder
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_2D, transmission_disorder_vals, 'g-', label='Disordered')
plt.xlabel(r'$k_y$')
plt.ylabel('Transmission')
plt.title(f'Transmission vs $k_y$ at E = {E_transport} with Disorder (W = {disorder_strength})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Function to add disorder to the Harper Hamiltonian
def harper_with_disorder(N, alpha, ky, disorder_strength=0.5, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        onsite = 2 * t * np.cos(phase)
        disorder = disorder_strength * (2 * np.random.rand() - 1)  # Uniform [-1, 1]
        H[m, m] = onsite + disorder
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Function to compute transmission with disorder
def transmission_disorder(N, alpha, ky, E, disorder_strength=1.0, t=1.0, eta=1e-3):
    H = harper_with_disorder(N, alpha, ky, disorder_strength, t)
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))

    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(N) - H)

    T = np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)
    return T

# Sweep over ky for a single realization of disorder
disorder_strength = 1.0  # Strong disorder
transmission_disorder_vals = []

for ky in tqdm(ky_vals_2D, desc="Transmission with Disorder"):
    T_dis = transmission_disorder(N, alpha, ky, E_transport, disorder_strength)
    transmission_disorder_vals.append(T_dis)

# Plot transmission with disorder
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_2D, transmission_disorder_vals, 'g-', label='Disordered')
plt.xlabel(r'$k_y$')
plt.ylabel('Transmission')
plt.title(f'Transmission vs $k_y$ at E = {E_transport} with Disorder (W = {disorder_strength})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Average transmission over multiple disorder realizations
num_realizations = 10
transmission_avg = np.zeros_like(ky_vals_2D)

for r in tqdm(range(num_realizations), desc="Averaging over disorder"):
    transmission_realization = []
    for ky in ky_vals_2D:
        T_dis = transmission_disorder(N, alpha, ky, E_transport, disorder_strength)
        transmission_realization.append(T_dis)
    transmission_avg += np.array(transmission_realization)

# Compute average
transmission_avg /= num_realizations

# Plot average transmission
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_2D, transmission_avg, 'b-', label=f'Avg over {num_realizations} realizations')
plt.xlabel(r'$k_y$')
plt.ylabel('Average Transmission')
plt.title(f'Average Transmission vs $k_y$ at E = {E_transport} with Disorder (W = {disorder_strength})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fix missing import
from fractions import Fraction

# Re-run Hofstadter butterfly visualization with Chern band tracking
alpha_vals = np.linspace(0, 1, 1000, endpoint=False)
E_vals_butterfly = []

for alpha in tqdm(alpha_vals, desc="Generating butterfly with Chern numbers"):
    frac = Fraction(alpha).limit_denominator(20)
    p, q = frac.numerator, frac.denominator
    alpha = p / q  # ensure rational
    for kx in np.linspace(-np.pi, np.pi, 40):
        H = np.zeros((q, q), dtype=np.complex128)
        for m in range(q):
            phase_y = 2 * np.pi * alpha * m
            H[m, m] = 2 * np.cos(phase_y)
            H[m, (m+1)%q] = -np.exp(-1j * kx)
            H[(m+1)%q, m] = -np.exp(1j * kx)
        eigvals = np.linalg.eigvalsh(H)
        E_vals_butterfly.extend([(alpha, E) for E in eigvals])

# Plot butterfly spectrum
alphas_plot, energies_plot = zip(*E_vals_butterfly)
plt.figure(figsize=(10, 6))
plt.scatter(alphas_plot, energies_plot, s=0.2, color='black')
plt.xlabel(r'$\alpha = \phi/\phi_0$')
plt.ylabel('Energy')
plt.title('Hofstadter Butterfly Spectrum')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize Berry curvature for a specific band over k-space for alpha = 1/3

alpha_bc = 1/3
frac = Fraction(alpha_bc).limit_denominator()
q = frac.denominator
Nk = 100  # finer grid for Berry curvature
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
berry_curvature = np.zeros((Nk, Nk))

# Compute Berry curvature for band 0
for i in range(Nk):
    for j in range(Nk):
        kx = kx_vals[i]
        ky = ky_vals[j]
        H = harper_magnetic_hamiltonian(kx, ky, alpha_bc)
        _, eigvecs = eigh(H)

        # Forward directions
        kx_next = kx_vals[(i+1)%Nk]
        ky_next = ky_vals[(j+1)%Nk]

        _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha_bc))
        _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha_bc))
        _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha_bc))

        band = 0  # Choose band index

        U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
        U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
        U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
        U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
        F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)  # Numerical stability
        berry_curvature[i, j] = F12.imag

# Normalize and sum for Hall conductance
chern = np.sum(berry_curvature) / (2 * np.pi)
sigma_xy = chern * (1.0 / (2 * np.pi))  # e^2/h = 1 unit

# Plot Berry curvature heatmap
plt.figure(figsize=(6, 5))
plt.imshow(berry_curvature.T, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
           cmap='RdBu', aspect='auto')
plt.colorbar(label='Berry Curvature')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature for Band 0 at $\alpha = {alpha_bc}$')
plt.tight_layout()
plt.show()

chern, sigma_xy


In [ ]:
# Re-define the harper_magnetic_hamiltonian function for use in Berry curvature plot
def harper_magnetic_hamiltonian(kx, ky, alpha=1, t=1.0):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase_y)
        H[m, (m+1)%q] = -t * np.exp(-1j * kx)
        H[(m+1)%q, m] = -t * np.exp(1j * kx)
    return H

# Re-run Berry curvature and conductance computation
berry_curvature = np.zeros((Nk, Nk))

for i in range(Nk):
    for j in range(Nk):
        kx = kx_vals[i]
        ky = ky_vals[j]
        H = harper_magnetic_hamiltonian(kx, ky, alpha_bc)
        _, eigvecs = eigh(H)

        kx_next = kx_vals[(i+1)%Nk]
        ky_next = ky_vals[(j+1)%Nk]

        _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha_bc))
        _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha_bc))
        _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha_bc))

        band = 0
        U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
        U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
        U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
        U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
        F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
        berry_curvature[i, j] = F12.imag

# Normalize and integrate
chern = np.sum(berry_curvature) / (2 * np.pi)
sigma_xy = chern * (1.0 / (2 * np.pi))

# Plot Berry curvature
plt.figure(figsize=(6, 5))
plt.imshow(berry_curvature.T, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
           cmap='RdBu', aspect='auto')
plt.colorbar(label='Berry Curvature')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature for Band 0 at $\alpha = {alpha_bc}$')
plt.tight_layout()
plt.show()

chern, sigma_xy


In [ ]:
# Compute and visualize Berry curvature for all bands at alpha = 1/3

berry_curvature_all = np.zeros((q, Nk, Nk))  # shape: (bands, kx, ky)

for i in range(Nk):
    for j in range(Nk):
        kx = kx_vals[i]
        ky = ky_vals[j]
        H = harper_magnetic_hamiltonian(kx, ky, alpha_bc)
        _, eigvecs = eigh(H)

        kx_next = kx_vals[(i+1)%Nk]
        ky_next = ky_vals[(j+1)%Nk]

        _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha_bc))
        _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha_bc))
        _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha_bc))

        for band in range(q):
            U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
            berry_curvature_all[band, i, j] = F12.imag

# Plot Berry curvature for each band
fig, axes = plt.subplots(1, q, figsize=(5 * q, 5 * p), sharey=True)
for band in range(q):
    ax = axes[band]
    im = ax.imshow(berry_curvature_all[band].T, origin='lower',
                   extent=[-np.pi, np.pi, -np.pi, np.pi],
                   cmap='RdBu', aspect='auto')
    ax.set_title(rf'Band {band} at $\alpha = {alpha_bc}$')
    ax.set_xlabel(r'$k_x$')
    if band == 0:
        ax.set_ylabel(r'$k_y$')
fig.colorbar(im, ax=axes, orientation='vertical', label='Berry Curvature')
plt.suptitle(r'Berry Curvature Across Bands at $\alpha = 1/3$', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Fix the layout warning by using constrained_layout instead of tight_layout

fig, axes = plt.subplots(1, q, figsize=(5 * q, 5), sharey=True, constrained_layout=True)
for band in range(q):
    ax = axes[band]
    im = ax.imshow(berry_curvature_all[band].T, origin='lower',
                   extent=[-np.pi, np.pi, -np.pi, np.pi],
                   cmap='RdBu', aspect='auto')
    ax.set_title(rf'Band {band} at $\alpha = {alpha_bc}$')
    ax.set_xlabel(r'$k_x$')
    if band == 0:
        ax.set_ylabel(r'$k_y$')

fig.colorbar(im, ax=axes, orientation='vertical', label='Berry Curvature')
fig.suptitle(r'Berry Curvature Across Bands at $\alpha = 1/3$', fontsize=14)
plt.show()


In [ ]:
# Simulate transmission T(ky, E) for Harper-Hofstadter model at alpha = 1/3
# using Green's function with self-energies (semi-infinite leads)

# System size and discretization
N = 50  # number of sites along x (finite width)
alpha_transport = 1/2
E_vals_trans = np.linspace(-np.pi, np.pi, 100)
ky_vals_trans = np.linspace(-np.pi, np.pi, 100)
T_kyE = np.zeros((len(ky_vals_trans), len(E_vals_trans)))

# Function to construct Harper model Hamiltonian with alpha and ky
def harper_strip(N, alpha, ky, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase)
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Function to compute transmission at fixed ky and E
def transmission_green(H, E, eta=1e-4, t=1.0):
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))
    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(H.shape[0]) - H)
    T = np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)
    return T

# Compute T(ky, E)
for i, ky in enumerate(ky_vals_trans):
    for j, E in enumerate(E_vals_trans):
        H = harper_strip(N, alpha_transport, ky)
        T_kyE[i, j] = transmission_green(H.copy(), E)

# Plot transmission map
plt.figure(figsize=(12, 10))
plt.imshow(T_kyE.T, origin='lower', extent=[-np.pi, np.pi, E_vals_trans[0], E_vals_trans[-1]],
           aspect='auto', cmap='inferno')
plt.colorbar(label='Transmission')
plt.xlabel(r'$k_y$')
plt.ylabel('Energy')
plt.title(r'Transmission $T(k_y, E)$ for $\alpha = \frac{1}{2}$')
plt.tight_layout()
plt.show()


In [ ]:
# Focused transmission calculation at a single energy across ky

E_focus = 0.5  # Fixed energy value
T_ky_focus = []

for ky in tqdm(ky_vals_trans, desc=f"Computing T(ky) at E={E_focus}"):
    H = harper_strip(N, alpha_transport, ky)
    T_val = transmission_green(H.copy(), E_focus)
    T_ky_focus.append(T_val)

# Plot T(ky) at fixed E
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_trans, T_ky_focus, 'b-')
plt.xlabel(r'$k_y$')
plt.ylabel('Transmission')
plt.title(rf'Transmission $T(k_y)$ at $E = {E_focus}$ for $\alpha = \frac{{1}}{{3}}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.constants import k, h, e
from scipy.special import expit  # logistic function for Fermi-Dirac

# Constants (in natural units: e = h = 1)
beta = 1 / (0.05)  # inverse temperature, T = 0.05 in units of hopping t

# Fermi-Dirac weight function (derivative)
def fermi_derivative(E, mu, beta):
    return -beta * expit(beta * (E - mu)) * (1 - expit(beta * (E - mu)))

# Energy range and Fermi level
E_vals_finiteT = np.linspace(-3, 3, 200)
mu = 0.5 # Fermi level (chemical potential)

# Calculate T(E, ky) and integrate over energy
T_integrated = []

for ky in tqdm(ky_vals_trans, desc="Integrating T(E, ky) over E"):
    T_E = np.array([transmission_green(harper_strip(N, alpha_transport, ky), E) for E in E_vals_finiteT])
    weight = fermi_derivative(E_vals_finiteT, mu, beta)
    G_ky = np.trapz(T_E * weight, E_vals_finiteT)
    T_integrated.append(G_ky)

# Plot integrated conductance vs ky
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_trans, T_integrated, 'r-')
plt.xlabel(r'$k_y$')
plt.ylabel(r'$G(k_y)$ (Conductance)')
plt.title(rf'Finite-Temperature Conductance $G(k_y)$ at $\mu = {mu}$, $T = {1/beta:.2f}$, $\alpha = \frac{{1}}{{3}}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Choose a single ky value (e.g., center of the Brillouin zone)
ky_single = 0.0  # can also try np.pi/3, etc.

# Compute T(E) for this ky
T_E_single = np.array([
    transmission_green(harper_strip(N, alpha_transport, ky_single), E) for E in E_vals_finiteT
])

# Fermi-Dirac weighting
weights = fermi_derivative(E_vals_finiteT, mu, beta)

# Integrate over energy
G_single_ky = np.trapz(T_E_single * weights, E_vals_finiteT)

# Plot T(E) and Fermi weight
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(E_vals_finiteT, T_E_single, 'b-', label='Transmission $T(E)$')
ax1.set_xlabel('Energy')
ax1.set_ylabel('Transmission', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(E_vals_finiteT, weights, 'r--', label="−dF/dE", alpha=0.6)
ax2.set_ylabel('Fermi Weight', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title(rf'Transport Integrand and Conductance at $k_y = {ky_single}$')
fig.tight_layout()
plt.show()

G_single_ky


In [ ]:
# Try a ky value where edge states were more prominent (e.g., near ky = pi/2)
ky_edge = 0.9*np.pi / 2

# Compute T(E) for this ky
T_E_edge = np.array([
    transmission_green(harper_strip(N, alpha_transport, ky_edge), E) for E in E_vals_finiteT
])

# Fermi-Dirac weighting
weights_edge = fermi_derivative(E_vals_finiteT, mu, beta)

# Integrate over energy
G_edge_ky = np.trapz(T_E_edge * weights_edge, E_vals_finiteT)

# Plot T(E) and Fermi weight
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(E_vals_finiteT, T_E_edge, 'b-', label='Transmission $T(E)$')
ax1.set_xlabel('Energy')
ax1.set_ylabel('Transmission', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(E_vals_finiteT, weights_edge, 'r--', label="−dF/dE", alpha=0.6)
ax2.set_ylabel('Fermi Weight', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title(rf'Transport Integrand and Conductance at $k_y = {ky_edge:.2f}$')
fig.tight_layout()
plt.show()

G_edge_ky


In [ ]:
# Coarse grid over ky for integrated conductance G(ky)
ky_vals_coarse = np.linspace(-np.pi, np.pi, 30)
G_ky_coarse = []

for ky in tqdm(ky_vals_coarse, desc="Computing G(ky) over coarse grid"):
    T_E = np.array([
        transmission_green(harper_strip(N, alpha_transport, ky), E) for E in E_vals_finiteT
    ])
    weight = fermi_derivative(E_vals_finiteT, mu, beta)
    G_val = np.trapz(T_E * weight, E_vals_finiteT)
    G_ky_coarse.append(G_val)

# Plot G(ky)
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_coarse, G_ky_coarse, 'm-o')
plt.xlabel(r'$k_y$')
plt.ylabel(r'$G(k_y)$ (Conductance)')
plt.title(rf'Edge Transport Conductance $G(k_y)$ at $\mu = {mu}$, $T = {1/beta:.2f}$, $\alpha = \frac{{1}}{{3}}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
!pip install pandas ace-tools as tools

In [ ]:
# Define 3–5 representative ky values: edge-like, center (bulk), and others
ky_sampled = [-np.pi/2, 0.0, np.pi/3, np.pi/2, np.pi]
G_ky_sampled = []

# Reduce energy resolution for speed
E_vals_faster = np.linspace(-2, 2, 50)  # tighter window around mu = 0.5

for ky in tqdm(ky_sampled, desc="Computing G(ky) for key points"):
    T_E = np.array([
        transmission_green(harper_strip(N, alpha_transport, ky), E) for E in E_vals_faster
    ])
    weight = fermi_derivative(E_vals_faster, mu, beta)
    G_val = np.trapz(T_E * weight, E_vals_faster)
    G_ky_sampled.append(G_val)

# Plot the sampled G(ky) data
plt.figure(figsize=(8, 5))
plt.plot(ky_sampled, G_ky_sampled, 'co-', linewidth=2, markersize=6)
plt.xlabel(r'$k_y$')
plt.ylabel(r'$G(k_y)$ (Conductance in $e^2/h$)')
plt.title(r'Sampled Finite-Temperature Conductance $G(k_y)$ at $\mu = 0.5$, $\alpha = \frac{1}{3}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Option 2: Expand the grid for smoother resolution (fewer points, faster energy integration)
ky_vals_smooth = np.linspace(-np.pi, np.pi, 20)
G_ky_smooth = []

# Use a tighter energy window and lower resolution for speed
E_vals_fast = np.linspace(0, 1, 60)  # around mu = 0.5

for ky in tqdm(ky_vals_smooth, desc="Expanded G(ky) scan"):
    T_E = np.array([
        transmission_green(harper_strip(N, alpha_transport, ky), E) for E in E_vals_fast
    ])
    weight = fermi_derivative(E_vals_fast, mu, beta)
    G_val = np.trapezoid(T_E * weight, E_vals_fast)
    G_ky_smooth.append(G_val)

# Plot the smoother G(ky) curve
plt.figure(figsize=(8, 5))
plt.plot(ky_vals_smooth, G_ky_smooth, 'm-', linewidth=2)
plt.xlabel(r'$k_y$')
plt.ylabel(r'$G(k_y)$ (Conductance in $e^2/h$)')
plt.title(r'Smoothed Finite-Temperature Conductance $G(k_y)$ at $\mu = 0.5$, $\alpha = \frac{1}{3}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# For overlay, we use the already computed Berry curvature from band 0 (index 0)
# Average Berry curvature over kx for each ky to compare with G(ky)

berry_band0 = berry_curvature_all[0]  # shape: (Nk, Nk)
Nk_used = berry_band0.shape[0]
ky_vals_bc = np.linspace(-np.pi, np.pi, Nk_used, endpoint=False)

# Average over kx direction to get 1D profile vs ky
berry_avg_ky = np.mean(berry_band0, axis=0)

# Interpolate smoothed G(ky) to match the ky grid of Berry curvature
from scipy.interpolate import interp1d

interp_G = interp1d(ky_vals_smooth, G_ky_smooth, kind='cubic', bounds_error=False, fill_value=0)
G_interp_bc = interp_G(ky_vals_bc)

# Plot combined visualization
fig, ax1 = plt.subplots(figsize=(10, 5))

# Berry curvature (avg over kx)
ax1.plot(ky_vals_bc, berry_avg_ky, 'b-', label='Avg Berry Curvature (Band 0)')
ax1.set_xlabel(r'$k_y$')
ax1.set_ylabel('Avg Berry Curvature', color='b')
ax1.tick_params(axis='y', labelcolor='b')

# Conductance overlay
ax2 = ax1.twinx()
ax2.plot(ky_vals_bc, G_interp_bc, 'r--', label='Conductance $G(k_y)$')
ax2.set_ylabel(r'Conductance $G(k_y)$ [$e^2/h$]', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title(r'Comparison of Avg Berry Curvature and Conductance $G(k_y)$ at $\alpha = \frac{1}{3}$')
fig.tight_layout()
plt.show()


In [ ]:
# Repeat the process for band 1 and band 2
fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

for idx, band in enumerate([1, 2]):
    berry_band = berry_curvature_all[band]
    berry_avg_ky_band = np.mean(berry_band, axis=0)

    ax1 = axes[idx]
    ax1.plot(ky_vals_bc, berry_avg_ky_band, 'b-', label=f'Avg Berry Curvature (Band {band})')
    ax1.set_ylabel('Avg Berry Curvature', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.legend(loc='upper left')

    ax2 = ax1.twinx()
    ax2.plot(ky_vals_bc, G_interp_bc, 'r--', label='Conductance $G(k_y)$')
    ax2.set_ylabel(r'Conductance $G(k_y)$ [$e^2/h$]', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    ax2.legend(loc='upper right')

axes[-1].set_xlabel(r'$k_y$')
fig.suptitle(r'Berry Curvature vs Conductance $G(k_y)$ for Bands 1 and 2 at $\alpha = \frac{1}{3}$', fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.animation as animation

# Define range of alpha values for animation
alpha_vals_anim = np.linspace(0, 1, 10)
Nk_anim = 30
kx_vals_anim = np.linspace(-np.pi, np.pi, Nk_anim, endpoint=False)
ky_vals_anim = np.linspace(-np.pi, np.pi, Nk_anim, endpoint=False)

# Prepare figure
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(np.zeros((Nk_anim, Nk_anim)), origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
               cmap='RdBu', aspect='auto', vmin=-np.pi, vmax=np.pi)
title = ax.set_title("")
ax.set_xlabel(r"$k_x$")
ax.set_ylabel(r"$k_y$")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Berry Curvature")

# Animation function
def update(frame):
    alpha = alpha_vals_anim[frame]
    q = Fraction(alpha).limit_denominator(20).denominator
    berry_curv = np.zeros((Nk_anim, Nk_anim))
    for i, kx in enumerate(kx_vals_anim):
        for j, ky in enumerate(ky_vals_anim):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            # Forward k-points
            kx_next = kx_vals_anim[(i+1)%Nk_anim]
            ky_next = ky_vals_anim[(j+1)%Nk_anim]
            _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha))
            _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha))
            _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha))

            band = 0  # Focus on lowest band
            U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
            berry_curv[i, j] = F12.imag

    im.set_data(berry_curv.T)
    title.set_text(rf"Berry Curvature (Band 0), $\alpha = {alpha:.3f}$")
    return [im, title]

# Create animation
ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals_anim), interval=800, blit=False)
plt.close(fig)

# Display animation
from IPython.display import HTML
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from fractions import Fraction

from mpl_toolkits.mplot3d import Axes3D

# Parameters
alpha = 1/2
band = 1
Nk = 50
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')
berry_curv = np.zeros_like(kx_grid)

# Define Hamiltonian
def harper_magnetic_hamiltonian(kx, ky, alpha=1, t=1.0):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase_y)
        H[m, (m+1)%q] = -t * np.exp(-1j * kx)
        H[(m+1)%q, m] = -t * np.exp(1j * kx)
    return H

# Compute Berry curvature
for i, kx in enumerate(kx_vals):
    for j, ky in enumerate(ky_vals):
        H = harper_magnetic_hamiltonian(kx, ky, alpha)
        _, eigvecs = eigh(H)

        kx_next = kx_vals[(i+1)%Nk]
        ky_next = ky_vals[(j+1)%Nk]
        _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha))
        _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha))
        _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha))

        U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
        U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
        U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
        U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
        F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
        berry_curv[i, j] = F12.imag

# Plot 3D surface
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(kx_grid, ky_grid, berry_curv, cmap='RdBu', edgecolor='k', linewidth=0.3)
ax.set_xlabel(r'$k_x$')
ax.set_ylabel(r'$k_y$')
ax.set_zlabel('Berry Curvature')
ax.set_title(rf'3D Berry Curvature for Band {band} at $\alpha = {alpha}$')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction

from mpl_toolkits.mplot3d import Axes3D

# Grid setup
Nk = 30
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# α values to animate over
alpha_vals = np.linspace(1/6, 1/2, 10)  # adjust number of frames as needed
band = 0

# Define Harper-Hofstadter Hamiltonian
def harper_magnetic_hamiltonian(kx, ky, alpha=1/3, t=1.0):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase_y)
        H[m, (m+1)%q] = -t * np.exp(-1j * kx)
        H[(m+1)%q, m] = -t * np.exp(1j * kx)
    return H

# Set up the figure and 3D axis
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = [ax.plot_surface(kx_grid, ky_grid, np.zeros_like(kx_grid), cmap='RdBu')]

ax.set_zlim(-np.pi, np.pi)
ax.set_xlabel(r'$k_x$')
ax.set_ylabel(r'$k_y$')
ax.set_zlabel('Berry Curvature')
title = ax.set_title("")

# Animation function
def update(frame):
    alpha = alpha_vals[frame]
    q = Fraction(alpha).limit_denominator(20).denominator
    berry_curv = np.zeros_like(kx_grid)
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)

            kx_next = kx_vals[(i+1)%Nk]
            ky_next = ky_vals[(j+1)%Nk]
            _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx_next, ky, alpha))
            _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky_next, alpha))
            _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx_next, ky_next, alpha))

            U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
            berry_curv[i, j] = F12.imag

    ax.clear()
    ax.plot_surface(kx_grid, ky_grid, berry_curv, cmap='RdBu')
    ax.set_zlim(-np.pi, np.pi)
    ax.set_xlabel(r'$k_x$')
    ax.set_ylabel(r'$k_y$')
    ax.set_zlabel('Berry Curvature')
    ax.set_title(rf'Berry Curvature (Band {band}) at $\alpha = {alpha:.3f}$')
    return ax,

# Build and display the animation
ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals), interval=800, blit=False)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction
from mpl_toolkits.mplot3d import Axes3D

# Discretization
Nk = 30
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# α values to animate
alpha_vals = np.linspace(0, 1, 5)
band_indices = [0, 1, 2]

# Set up figure and axis
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = [ax.plot_surface(kx_grid, ky_grid, np.zeros_like(kx_grid), cmap='RdBu')]
title = ax.set_title("")

# Animation update function
def update(frame):
    alpha_index = frame // len(band_indices)
    band_index = band_indices[frame % len(band_indices)]
    alpha = alpha_vals[alpha_index]
    q = Fraction(alpha).limit_denominator(20).denominator
    if band_index >= q:
        ax.clear()
        ax.set_title(f"No band {band_index} at α = {alpha:.3f}")
        return ax,

    berry_curv = np.zeros_like(kx_grid)
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            def H_k(kx, ky): return harper_magnetic_hamiltonian(kx, ky, alpha)
            H = H_k(kx, ky)
            _, eigvecs = eigh(H)
            kx1, ky1 = kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk]
            _, vecs_kx = eigh(H_k(kx1, ky))
            _, vecs_ky = eigh(H_k(kx, ky1))
            _, vecs_diag = eigh(H_k(kx1, ky1))

            U1 = np.vdot(eigvecs[:, band_index], vecs_kx[:, band_index])
            U2 = np.vdot(vecs_kx[:, band_index], vecs_diag[:, band_index])
            U3 = np.vdot(vecs_diag[:, band_index], vecs_ky[:, band_index])
            U4 = np.vdot(vecs_ky[:, band_index], eigvecs[:, band_index])
            F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
            berry_curv[i, j] = F12.imag

    ax.clear()
    ax.plot_surface(kx_grid, ky_grid, berry_curv, cmap='RdBu')
    ax.set_zlim(-np.pi, np.pi)
    ax.set_xlabel(r'$k_x$')
    ax.set_ylabel(r'$k_y$')
    ax.set_zlabel('Berry Curvature')
    ax.set_title(rf'Band {band_index}, $\alpha = {alpha:.3f}$')
    return ax,

# Hamiltonian builder
def harper_magnetic_hamiltonian(kx, ky, alpha=1/3, t=1.0):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase_y)
        H[m, (m+1)%q] = -t * np.exp(-1j * kx)
        H[(m+1)%q, m] = -t * np.exp(1j * kx)
    return H

# Total frames = len(alpha_vals) * len(band_indices)
ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals) * len(band_indices),
                              interval=1000, blit=False)
plt.show()
ani.save('berry_curv.mp4')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction

# Setup
Nk = 30
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')
E_vals = np.linspace(0, 1, 60)
mu = 0.5
beta = 1 / 0.05  # T = 0.05
eta = 1e-3
N = 30  # Width of the strip for transport
alpha_vals = np.linspace(1/6, 1/2, 6)

# Harper strip for transport
def harper_strip(N, alpha, ky, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase)
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Surface self-energy for leads
def surface_self_energy(t, E, eta=1e-3):
    z = E + 1j * eta
    return (z - np.sqrt(z**2 - 4 * t**2)) / 2

# Conductance integrand weight (Fermi derivative)
def fermi_derivative(E, mu, beta):
    from scipy.special import expit
    return -beta * expit(beta * (E - mu)) * (1 - expit(beta * (E - mu)))

# Transmission via Green's function
def transmission_green(H, E, eta=1e-3, t=1.0):
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))
    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(H.shape[0]) - H)
    return np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)

# Harper Hamiltonian for Berry curvature
def harper_magnetic_hamiltonian(kx, ky, alpha=1/3, t=1.0):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase_y)
        H[m, (m+1)%q] = -t * np.exp(-1j * kx)
        H[(m+1)%q, m] = -t * np.exp(1j * kx)
    return H

# Setup figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
berry_plot = ax1.imshow(np.zeros_like(kx_grid), origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
                        cmap='RdBu', aspect='auto', vmin=-np.pi, vmax=np.pi)
G_plot, = ax2.plot([], [], 'r-')
ax1.set_title("Berry Curvature (Band 0)")
ax1.set_xlabel(r"$k_x$")
ax1.set_ylabel(r"$k_y$")
ax2.set_title("Conductance vs $k_y$")
ax2.set_xlabel(r"$k_y$")
ax2.set_ylabel(r"$G(k_y)$ [$e^2/h$]")
ax2.set_ylim(0, 1)

title = fig.suptitle("")

def update(frame):
    alpha = alpha_vals[frame]
    q = Fraction(alpha).limit_denominator(20).denominator
    berry_curv = np.zeros_like(kx_grid)
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            band = 0

            kx1, ky1 = kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk]
            _, vecs_kx = eigh(harper_magnetic_hamiltonian(kx1, ky, alpha))
            _, vecs_ky = eigh(harper_magnetic_hamiltonian(kx, ky1, alpha))
            _, vecs_diag = eigh(harper_magnetic_hamiltonian(kx1, ky1, alpha))

            U1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            U2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            U3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            U4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(U1 * U2 * U3 * U4 + 1e-20)
            berry_curv[i, j] = F12.imag

    berry_plot.set_data(berry_curv.T)

    # Conductance G(ky)
    ky_vals = np.linspace(-np.pi, np.pi, 40)
    G_vals = []
    for ky in ky_vals:
        H_strip = harper_strip(N, alpha, ky)
        T_E = np.array([transmission_green(H_strip.copy(), E) for E in E_vals])
        weight = fermi_derivative(E_vals, mu, beta)
        G_vals.append(np.trapz(T_E * weight, E_vals))

    G_plot.set_data(ky_vals, G_vals)
    ax2.set_xlim(ky_vals[0], ky_vals[-1])
    title.set_text(rf"$\alpha = {alpha:.3f}$")

    return [berry_plot, G_plot, title]

ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals), interval=100, blit=False)

plot.show()
ani.save('berry_curv2.mp4')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction
from IPython.display import HTML

# Basic setup
Nk = 20
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')
alpha_vals = np.linspace(1/6, 1/2, 6)
band = 0

# Hamiltonian
def harper_magnetic_hamiltonian(kx, ky, alpha):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase_y)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Setup plot
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(np.zeros((Nk, Nk)), origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
                  cmap='RdBu', vmin=-np.pi, vmax=np.pi, animated=True)
title = ax.set_title("")

def update(frame):
    alpha = alpha_vals[frame]
    berry = np.zeros((Nk, Nk))
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            berry[i, j] = F12.imag

    image.set_array(berry.T)
    title.set_text(f"Berry Curvature, α = {alpha:.3f}")
    return [image, title]

ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals), interval=1000, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction
from IPython.display import HTML

# Grid setup
Nk = 20
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals_arr = np.linspace(-np.pi, np.pi, Nk, endpoint=False)  # renamed to avoid shadowing
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals_arr, indexing='ij')
alpha_vals = np.linspace(1/6, 1/2, 6)
band = 0

# Harper Hamiltonian
def harper_magnetic_hamiltonian(kx, ky, alpha):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase_y)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Setup figure
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(np.zeros((Nk, Nk)), origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
                  cmap='RdBu', vmin=-np.pi, vmax=np.pi, animated=True)
title = ax.set_title("")

def update(frame):
    alpha = alpha_vals[frame]
    berry = np.zeros((Nk, Nk))
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals_arr):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals_arr[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals_arr[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            berry[i, j] = F12.imag

    image.set_array(berry.T)
    title.set_text(f"Berry Curvature, α = {alpha:.3f}")
    return [image, title]

ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals), interval=1000, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

# Grid setup
Nk = 20
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')
alpha_vals = np.linspace(1/6, 1/2, 6)
band = 0

# Harper Hamiltonian
def harper_magnetic_hamiltonian(kx, ky, alpha):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase_y)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Setup figure for 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(kx_grid, ky_grid, np.zeros_like(kx_grid), cmap='RdBu', edgecolor='k')
ax.set_zlim(-np.pi, np.pi)
ax.set_xlabel(r'$k_x$')
ax.set_ylabel(r'$k_y$')
ax.set_zlabel('Berry Curvature')
title = ax.set_title("")

def update(frame):
    alpha = alpha_vals[frame]
    berry = np.zeros((Nk, Nk))
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            berry[i, j] = F12.imag

    ax.clear()
    ax.plot_surface(kx_grid, ky_grid, berry.T, cmap='RdBu', edgecolor='k')
    ax.set_zlim(-np.pi, np.pi)
    ax.set_xlabel(r'$k_x$')
    ax.set_ylabel(r'$k_y$')
    ax.set_zlabel('Berry Curvature')
    ax.set_title(rf'Band {band}, $\alpha = {alpha:.3f}$')
    return ax,

ani = animation.FuncAnimation(fig, update, frames=len(alpha_vals), interval=10000, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.linalg import eigh
from fractions import Fraction
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

# Grid and α setup
Nk = 20
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')
alpha_vals = np.linspace(1/6, 1/2, 4)  # small set for speed
band_list = [0, 1, 2]  # animate up to 3 bands

# Build frames: all (alpha, band) pairs that are valid
frames = []
for alpha in alpha_vals:
    q = Fraction(alpha).limit_denominator(20).denominator
    for band in range(min(q, len(band_list))):
        frames.append((alpha, band))

# Hamiltonian builder
def harper_magnetic_hamiltonian(kx, ky, alpha):
    q = Fraction(alpha).limit_denominator(20).denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase_y)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Set up 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_zlim(-np.pi, np.pi)
ax.set_xlabel(r'$k_x$')
ax.set_ylabel(r'$k_y$')
ax.set_zlabel('Berry Curvature')
title = ax.set_title("")

# Update function
def update(frame_index):
    alpha, band = frames[frame_index]
    berry = np.zeros((Nk, Nk))
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            berry[i, j] = F12.imag

    ax.clear()
    ax.plot_surface(kx_grid, ky_grid, berry.T, cmap='RdBu', edgecolor='k')
    ax.set_zlim(-np.pi, np.pi)
    ax.set_xlabel(r'$k_x$')
    ax.set_ylabel(r'$k_y$')
    ax.set_zlabel('Berry Curvature')
    ax.set_title(rf'Band {band}, $\alpha = {alpha:.3f}$')
    return ax,
    # Extend animation to include Chern number label and prepare for Hofstadter spectrum overlay
def compute_chern_number(alpha, band, Nk=20):
    kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
    ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
    total_flux = 0.0
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            total_flux += F12.imag
    chern = total_flux / (2 * np.pi)
    return round(chern)

# Update function with Chern number annotation
def update_with_chern(frame_index):
    alpha, band = frames[frame_index]
    berry = np.zeros((Nk, Nk))
    for i, kx in enumerate(kx_vals):
        for j, ky in enumerate(ky_vals):
            H = harper_magnetic_hamiltonian(kx, ky, alpha)
            _, eigvecs = eigh(H)
            H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha)
            H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha)
            H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha)
            _, vecs_kx = eigh(H_kx)
            _, vecs_ky = eigh(H_ky)
            _, vecs_diag = eigh(H_diag)

            u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
            u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
            u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
            u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
            F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
            berry[i, j] = F12.imag

    chern = compute_chern_number(alpha, band)
    ax.clear()
    ax.plot_surface(kx_grid, ky_grid, berry.T, cmap='RdBu', edgecolor='k')
    ax.set_zlim(-np.pi, np.pi)
    ax.set_xlabel(r'$k_x$')
    ax.set_ylabel(r'$k_y$')
    ax.set_zlabel('Berry Curvature')
    ax.set_title(rf'Band {band}, $\alpha = {alpha:.3f}$, Chern = {chern}')
    return ax,

# Rebuild animation with updated function
ani = animation.FuncAnimation(fig, update_with_chern, frames=len(frames), interval=1200, blit=False)
plt.close(fig)

# Display animated plot with Chern numbers
HTML(ani.to_jshtml())


# Animate
ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
# Redefine kx_grid and ky_grid since they were lost during previous clearing
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# Rebuild the 3D figure and animation
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Final animation call with everything defined
ani = animation.FuncAnimation(fig, update_with_chern, frames=len(frames), interval=1200, blit=False)
plt.close(fig)

# Display the animation with Chern number overlay
HTML(ani.to_jshtml())


In [ ]:
# Compute Hofstadter spectrum with overlaid conductance band markers

# Parameters for Hofstadter spectrum
kx_vals_spec = np.linspace(-np.pi, np.pi, 100)
alpha_vals_spec = np.linspace(0, 1, 100, endpoint=False)
spectrum = []

# For each alpha, compute eigenvalues at each kx
for alpha in tqdm(alpha_vals_spec, desc="Computing Hofstadter spectrum"):
    frac = Fraction(alpha).limit_denominator(17)
    p, q = frac.numerator, frac.denominator
    for kx in kx_vals_spec:
        H = np.zeros((q, q), dtype=np.complex128)
        for m in range(q):
            phase = 2 * np.pi * alpha * m
            H[m, m] = 2 * np.cos(phase)
            H[m, (m+1)%q] = -np.exp(-1j * kx)
            H[(m+1)%q, m] = -np.exp(1j * kx)
        eigvals = np.linalg.eigvalsh(H)
        spectrum.extend([(alpha, E) for E in eigvals])

# Convert to plottable arrays
alphas_plot, energies_plot = zip(*spectrum)

# Plot the Hofstadter butterfly
plt.figure(figsize=(10, 6))
plt.scatter(alphas_plot, energies_plot, s=0.2, color='black')
plt.xlabel(r'Flux $\alpha$')
plt.ylabel('Energy')
plt.title('Hofstadter Butterfly Spectrum with Conductance Context')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Compute G(E, alpha) heatmap: integrated conductance vs energy and flux

E_vals_heat = np.linspace(-np.pi, np.pi, 100)
alpha_vals_heat = np.linspace(0.05, 1, 100)
ky_vals_heat = np.linspace(-np.pi, np.pi, 100)
G_E_alpha = np.zeros((len(alpha_vals_heat), len(E_vals_heat)))

# Transport strip setup
def harper_strip(N, alpha, ky, t=1.0):
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * t * np.cos(phase)
        if m < N - 1:
            H[m, m+1] = -t
            H[m+1, m] = -t
    return H

# Compute surface self-energy
def surface_self_energy(t, E, eta=1e-3):
    z = E + 1j * eta
    return (z - np.sqrt(z**2 - 4 * t**2)) / 2

# Compute transmission
def transmission_green(H, E, eta=1e-4, t=1.0):
    sigma = surface_self_energy(t, E, eta)
    Gamma = 1j * (sigma - np.conj(sigma))
    H[0, 0] += sigma
    H[-1, -1] += sigma
    G = np.linalg.inv((E + 1j * eta) * np.eye(H.shape[0]) - H)
    return np.real(Gamma * np.abs(G[0, -1])**2 * Gamma)

# Populate the conductance heatmap
for i, alpha in enumerate(tqdm(alpha_vals_heat, desc="Computing G(E, alpha) map")):
    for j, E in enumerate(E_vals_heat):
        G_sum = 0
        for ky in ky_vals_heat:
            H = harper_strip(N=30, alpha=alpha, ky=ky)
            G_sum += transmission_green(H.copy(), E)
        G_E_alpha[i, j] = G_sum / len(ky_vals_heat)

# Plot the heatmap
plt.figure(figsize=(10, 6))
plt.imshow(G_E_alpha.T, extent=[alpha_vals_heat[0], alpha_vals_heat[-1], E_vals_heat[0], E_vals_heat[-1]],
           origin='lower', aspect='auto', cmap='plasma')
plt.colorbar(label='Average Conductance $G(E, \\alpha)$ [$e^2/h$]')
plt.xlabel('Flux $\\alpha$')
plt.ylabel('Energy')
plt.title('Conductance Map $G(E, \\alpha)$ Overlaying Hofstadter Bands')
plt.tight_layout()
plt.show()


In [ ]:
# Plot Hofstadter spectrum with a fixed Fermi energy line overlay

fermi_level = 0.0

plt.figure(figsize=(10, 6))
plt.scatter(alphas_plot, energies_plot, s=0.2, color='black', label='Hofstadter bands')
plt.axhline(fermi_level, color='red', linestyle='--', linewidth=1.5, label=rf'Fermi level $\mu = {fermi_level}$')
plt.xlabel(r'Flux $\alpha$')
plt.ylabel('Energy')
plt.title('Hofstadter Spectrum with Fermi Level Overlay')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Compare the Fermi level μ=0.5 with Berry curvature at alpha = 1/3

alpha_mu = 5/2
mu = 1.0
Nk = 11
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# Hamiltonian to compute eigenvalues at each (kx, ky)
def harper_magnetic_hamiltonian(kx, ky, alpha=1/3):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Find which band(s) are occupied at mu
q = Fraction(alpha_mu).limit_denominator().denominator
berry_curvature_mu = np.zeros((Nk, Nk))
for i, kx in enumerate(kx_vals):
    for j, ky in enumerate(ky_vals):
        H = harper_magnetic_hamiltonian(kx, ky, alpha_mu)
        eigvals, eigvecs = eigh(H)
        for band, E in enumerate(eigvals):
            if E < mu:
                # Compute Berry curvature contribution from occupied bands
                H_kx = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky, alpha_mu)
                H_ky = harper_magnetic_hamiltonian(kx, ky_vals[(j+1)%Nk], alpha_mu)
                H_diag = harper_magnetic_hamiltonian(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha_mu)
                _, vecs_kx = eigh(H_kx)
                _, vecs_ky = eigh(H_ky)
                _, vecs_diag = eigh(H_diag)

                u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
                u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
                u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
                u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
                F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
                berry_curvature_mu[i, j] += F12.imag

# Normalize and plot
plt.figure(figsize=(7, 6))
plt.imshow(berry_curvature_mu.T, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
           cmap='RdBu', aspect='auto')
plt.colorbar(label='Accumulated Berry Curvature (μ < 0.5)')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature from Occupied Bands at $\alpha = \frac{{2}}{{3}}$, $\mu = {mu}$')
plt.tight_layout()
plt.show()


In [ ]:
# Compare the Fermi level μ=0.5 with Berry curvature at alpha = 1/3

alpha_mu = 1/2
mu = 1.0
Nk = 10
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

def harper_with_electric_potential(N,kx, ky,alpha, alpha_mu, E_field=0.01):
    q = Fraction(E_field).limit_denominator().denominator
    H = np.zeros((N, N), dtype=np.complex128)
    for m in range(N):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase) + E_field * m  # Add Stark potential
        if m < N - 1:
            H[m, m+1] = -1
            H[m+1, m] = -1
    return H

# Hamiltonian to compute eigenvalues at each (kx, ky)
#def harper_magnetic_hamiltonian(kx, ky, alpha=1/2):
#    q = Fraction(alpha).limit_denominator().denominator
#    H = np.zeros((q, q), dtype=np.complex128)
#    for m in range(q):
#        phase = 2 * np.pi * alpha * m + ky
#        H[m, m] = 2 * np.cos(phase)
#        H[m, (m+1)%q] = -np.exp(-1j * kx)
#        H[(m+1)%q, m] = -np.exp(1j * kx)
#    return H

# Find which band(s) are occupied at mu
q = Fraction(alpha_mu).limit_denominator().denominator
berry_curvature_mu = np.zeros((Nk, Nk))
for i, kx in enumerate(kx_vals):
    for j, ky in enumerate(ky_vals):
        H = harper_with_electric_potential(kx, ky, alpha, alpha_mu)
        eigvals, eigvecs = eigh(H)
        for band, E in enumerate(eigvals):
            if E < mu:
                # Compute Berry curvature contribution from occupied bands
                H_kx = harper_with_electric_potential(kx_vals[(i+1)%Nk], ky, alpha_mu)
                H_ky = harper_with_electric_potential(kx, ky_vals[(j+1)%Nk], alpha_mu)
                H_diag = harper_with_electric_potential(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha_mu)
                _, vecs_kx = eigh(H_kx)
                _, vecs_ky = eigh(H_ky)
                _, vecs_diag = eigh(H_diag)

                u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
                u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
                u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
                u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
         # Re-import packages after code state reset
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from fractions import Fraction

# Re-define Berry curvature visualization under electric field modulation
E_field = 0.1  # strength of electric field
alpha_stark = 1/3
Nk = 30
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# Define Harper model with Stark potential (linear electric field)
def harper_magnetic_stark(kx, ky, alpha=1/3, E_field=0.0):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        stark_shift = E_field * m  # linear Stark potential
        H[m, m] = 2 * np.cos(phase_y) + stark_shift
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Compute Berry curvature with electric field
berry_stark = np.zeros((Nk, Nk))
band = 0

for i, kx in enumerate(kx_vals):
    for j, ky in enumerate(ky_vals):
        H = harper_magnetic_stark(kx, ky, alpha_stark, E_field=E_field)
        _, eigvecs = eigh(H)
        H_kx = harper_magnetic_stark(kx_vals[(i+1)%Nk], ky, alpha_stark, E_field=E_field)
        H_ky = harper_magnetic_stark(kx, ky_vals[(j+1)%Nk], alpha_stark, E_field=E_field)
        H_diag = harper_magnetic_stark(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha_stark, E_field=E_field)
        _, vecs_kx = eigh(H_kx)
        _, vecs_ky = eigh(H_ky)
        _, vecs_diag = eigh(H_diag)

        u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
        u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
        u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
        u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
        F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
        berry_stark[i, j] = F12.imag

# Plot Berry curvature with electric field modulation
plt.figure(figsize=(7, 6))
plt.imshow(berry_stark.T, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
           cmap='coolwarm', aspect='auto')
plt.colorbar(label='Berry Curvature (Stark-Shifted)')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature for Band 0 at $\alpha = \frac{{1}}{{3}}$, $E_\mathrm{{field}} = {E_field}$')
plt.tight_layout()
plt.show()
       F12 = np.log(u1 * u2 * u3 * u4 + 1e-20)
                berry_curvature_mu[i, j] += F12.imag

# Normalize and plot
plt.figure(figsize=(12, 10))
plt.imshow(berry_curvature_mu.T, origin='lower', extent=[-np.pi, np.pi, -np.pi, np.pi],
           cmap='RdBu', aspect='auto')
plt.colorbar(label='Accumulated Berry Curvature (μ < 0.5)')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature from Occupied Bands at $\alpha = \frac{{4}}{{3}}$, $\mu = {mu}$')
plt.tight_layout()
plt.show()


In [ ]:
def harper_magnetic_hamiltonian(kx, ky, alpha=1/3):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase = 2 * np.pi * alpha * m + ky
        H[m, m] = 2 * np.cos(phase)
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

In [ ]:
# Re-import packages after code state reset
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from fractions import Fraction

# Re-define Berry curvature visualization under electric field modulation
E_field = 0.9  # strength of electric field
alpha_stark = 4/3
Nk = 91
kx_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)
kx_grid, ky_grid = np.meshgrid(kx_vals, ky_vals, indexing='ij')

# Define Harper model with Stark potential (linear electric field)
def harper_magnetic_stark(kx, ky, alpha=1, E_field=1.0):
    q = Fraction(alpha).limit_denominator().denominator
    H = np.zeros((q, q), dtype=np.complex128)
    for m in range(q):
        phase_y = 2 * np.pi * alpha * m + ky
        stark_shift = E_field * m  # linear Stark potential
        H[m, m] = 2 * np.cos(phase_y) + stark_shift
        H[m, (m+1)%q] = -np.exp(-1j * kx)
        H[(m+1)%q, m] = -np.exp(1j * kx)
    return H

# Compute Berry curvature with electric field
berry_stark = np.zeros((Nk, Nk))
band = 0

for i, kx in enumerate(kx_vals):
    for j, ky in enumerate(ky_vals):
        H = harper_magnetic_stark(kx, ky, alpha_stark, E_field=E_field)
        _, eigvecs = eigh(H)
        H_kx = harper_magnetic_stark(kx_vals[(i+1)%Nk], ky, alpha_stark, E_field=E_field)
        H_ky = harper_magnetic_stark(kx, ky_vals[(j+1)%Nk], alpha_stark, E_field=E_field)
        H_diag = harper_magnetic_stark(kx_vals[(i+1)%Nk], ky_vals[(j+1)%Nk], alpha_stark, E_field=E_field)
        _, vecs_kx = eigh(H_kx)
        _, vecs_ky = eigh(H_ky)
        _, vecs_diag = eigh(H_diag)

        u1 = np.vdot(eigvecs[:, band], vecs_kx[:, band])
        u2 = np.vdot(vecs_kx[:, band], vecs_diag[:, band])
        u3 = np.vdot(vecs_diag[:, band], vecs_ky[:, band])
        u4 = np.vdot(vecs_ky[:, band], eigvecs[:, band])
        F12 = np.log(u1 * u2 * u3 * u4 + 1e-22)
        berry_stark[i, j] = F12.imag

# Plot Berry curvature with electric field modulation
plt.figure(figsize=(12, 10))
plt.imshow(berry_stark.T, origin='lower', extent=[-2*np.pi, np.pi, -2*np.pi, np.pi],
           cmap='coolwarm', aspect='auto')
plt.colorbar(label='Berry Curvature (Stark-Shifted)')
plt.xlabel(r'$k_x$')
plt.ylabel(r'$k_y$')
plt.title(rf'Berry Curvature for Band 0 at $\alpha = \frac{{1}}{{3}}$, $E_\mathrm{{field}} = {E_field}$')
plt.tight_layout()
plt.show()


In [ ]:
# Compute and plot the density of states (DOS) in the Landau gauge using a square lattice tight-binding model

# Parameters
Lx, Ly = 30, 30  # lattice size
alpha = 4/3  # magnetic flux per plaquette (φ = 2π*α)
Nk = 100  # number of ky points
eta = 0.05  # broadening for DOS

# Generate ky values (Landau gauge is periodic in y)
ky_vals = np.linspace(-np.pi, np.pi, Nk, endpoint=False)

# DOS collection
all_eigs = []

# Build tight-binding Hamiltonian with Peierls substitution in Landau gauge: A = (0, Bx, 0)
for ky in ky_vals:
    H = np.zeros((Lx, Lx), dtype=np.complex128)
    for x in range(Lx):
        # Onsite + hopping in x-direction (no phase)
        if x < Lx - 1:
            H[x, x+1] = -1
            H[x+1, x] = -1
        # Hopping in y-direction (adds phase dependent on x)
        phase = 2 * np.pi * alpha * x + ky
        H[x, x] += -2 * np.cos(phase)
    eigvals = np.linalg.eigvalsh(H)
    all_eigs.extend(eigvals)

# Compute histogram for DOS
E_vals = np.linspace(-4, 4, 500)
dos = np.zeros_like(E_vals)
for E in all_eigs:
    dos += eta / (np.pi * ((E_vals - E)**2 + eta**2))

# Plot DOS
plt.figure(figsize=(8, 5))
plt.plot(E_vals, dos, color='darkblue')
plt.xlabel('Energy')
plt.ylabel('DOS (a.u.)')
plt.title(r'Density of States in Magnetic Field (Landau Gauge), $\alpha = \frac{1}{6}$')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Parameters
s = 20e-9  # spacer in meters
L = 100e-9  # simulation size (200 nm x 200 nm)
N = 200  # grid resolution

# Define grid
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)

# Potential function: V(x, y) = cos(2πx/s) * cos(2πy/s)
V = np.cos(2 * np.pi * X / s) * np.cos(2 * np.pi * Y / s)

# Plot
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none')

ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential V(x, y)')
ax.set_title(r'3D Potential $V(x, y) = \cos(2\pi x/s) \cos(2\pi y/s)$ with $s = 20$ nm')
fig.colorbar(surf, shrink=0.6, aspect=10, label='Potential (a.u.)')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Parameters
s = 1e-9  # spacer in meters
L = 200e-9  # 200 nm x 200 nm area
N = 8
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)

# Potential function
V = np.cos(2 * np.pi * X / s) * np.cos(2 * np.pi * Y / s)

# Trap site coordinates (nm units for display)
trap_sites_nm = np.array([
    [0, 0],
    [20, 0],
    [40, 0],
    [0, 20],
    [20, 20],
    [40, 20],
    [20, 40]
])
trap_sites_m = trap_sites_nm * 1e-9  # convert to meters

# Evaluate potential at trap locations
trap_z = np.cos(2 * np.pi * trap_sites_m[:, 0] / s) * np.cos(2 * np.pi * trap_sites_m[:, 1] / s)

# Plot
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Surface plot
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none', alpha=0.9)

# Contour projection onto bottom
ax.contour(X * 1e9, Y * 1e9, V, zdir='z', offset=-1.2, cmap='viridis', linewidths=0.8)

# Trap site markers
ax.scatter(trap_sites_nm[:, 0], trap_sites_nm[:, 1], trap_z, color='black', s=40, label='Trap sites')

# Axes labels and colorbar
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential V(x, y)')
ax.set_title(r'3D Potential Surface with Contours and Trap Sites ($s = 20$ nm)')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=12, label='Potential (a.u.)')

ax.set_zlim(-1.2, 1.1)
plt.tight_layout()
plt.show()


In [ ]:
# Simulate a Gaussian wavefunction centered at one of the trap sites
# We'll use the central trap at (20 nm, 20 nm) as the wavefunction center

# Wavefunction parameters
x0_nm, y0_nm = 7, 7  # center in nm
sigma_nm = 3.5  # width of the wavefunction in nm
x0 = x0_nm * 1e-9
y0 = y0_nm * 1e-9
sigma = sigma_nm * 1e-9

# Gaussian wavefunction: normalized 2D Gaussian
psi = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
psi /= np.sqrt(np.sum(psi**2))  # normalize

# Plot potential surface with wavefunction overlay (translucent)
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Main potential surface
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none', alpha=0.9)

# Contour projection of potential
ax.contour(X * 1e9, Y * 1e9, V, zdir='z', offset=-1.2, cmap='viridis', linewidths=0.8)

# Wavefunction overlay: semi-transparent surface
wave_overlay = ax.plot_surface(X * 1e9, Y * 1e9, psi * 2 - 1.2, cmap='hot', alpha=0.5)

# Trap sites
ax.scatter(trap_sites_nm[:, 0], trap_sites_nm[:, 1], trap_z, color='black', s=40, label='Trap sites')

# Axis settings
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential / Wavefunction')
ax.set_title('Potential with Gaussian Wavefunction Overlay ($s = 20$ nm)')
ax.set_zlim(-1.2, 1.1)
fig.colorbar(surf, ax=ax, shrink=0.6, aspect=12, label='Potential (a.u.)')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.animation as animation

# Define time-evolving wavefunction center (moves diagonally)
num_frames = 60
x_path = np.linspace(10e-9, 90e-9, num_frames)
y_path = np.linspace(10e-9, 90e-9, num_frames)

# Setup figure and axis for animation
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
ax.set_zlim(-1.2, 1.1)
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential / Wavefunction')
ax.set_title('Animated Wavefunction in Periodic Potential')
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none', alpha=0.9)
contours = ax.contour(X * 1e9, Y * 1e9, V, zdir='z', offset=-1.2, cmap='viridis', linewidths=0.8)
wf_plot = [ax.plot_surface(X * 1e9, Y * 1e9, np.zeros_like(V), cmap='hot', alpha=0.5)]

# Animation update function
def update(frame):
    # Clear previous wavefunction
    wf_plot[0].remove()

    x0 = x_path[frame]
    y0 = y_path[frame]
    psi_frame = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
    psi_frame /= np.sqrt(np.sum(psi_frame**2))
    wf_plot[0] = ax.plot_surface(X * 1e9, Y * 1e9, psi_frame * 2 - 1.2,
                                 cmap='hot', alpha=0.5)
    return wf_plot[0],

# Build and display animation
ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=100, blit=False)
plt.close(fig)

from IPython.display import HTML
HTML(ani.to_jshtml())


In [ ]:
# Define multiple wavefunction centers
centers_nm = np.array([
    [20, 20],
    [60, 20],
    [40, 60],
    [80, 80]
])
centers = centers_nm * 1e-9  # convert to meters

# Compute the superposition of multiple localized wavefunctions
psi_multi = np.zeros_like(X)
for x0, y0 in centers:
    psi_component = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
    psi_multi += psi_component

# Normalize the total wavefunction
psi_multi /= np.sqrt(np.sum(psi_multi**2))

# Plot potential + multi-wavefunction overlay
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Base potential surface
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none', alpha=0.9)

# Contour projection
ax.contour(X * 1e9, Y * 1e9, V, zdir='z', offset=-1.2, cmap='viridis', linewidths=0.8)

# Overlay wavefunction as translucent red-orange surface
wave_overlay = ax.plot_surface(X * 1e9, Y * 1e9, psi_multi * 2 - 1.2,
                               cmap='hot', alpha=0.5)

# Trap site markers
ax.scatter(trap_sites_nm[:, 0], trap_sites_nm[:, 1], trap_z, color='black', s=40, label='Trap sites')

# Formatting
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential / Wavefunction')
ax.set_title('Multiple Localized Wavefunctions in Periodic Potential')
ax.set_zlim(-1.2, 1.1)
fig.colorbar(surf, ax=ax, shrink=0.6, aspect=12, label='Potential (a.u.)')

plt.tight_layout()
plt.show()


In [ ]:
# Define multiple wavefunction centers
centers_nm = np.array([
    [20, 20],
    [60, 20],
    [40, 60],
    [80, 80]
])
centers = centers_nm * 1e-9  # convert to meters

# Compute the superposition of multiple localized wavefunctions
psi_multi = np.zeros_like(X)
for x0, y0 in centers:
    psi_component = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
    psi_multi += psi_component

# Normalize the total wavefunction
psi_multi /= np.sqrt(np.sum(psi_multi**2))

# Plot potential + multi-wavefunction overlay
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Base potential surface
surf = ax.plot_surface(X * 1e9, Y * 1e9, V, cmap='viridis', edgecolor='none', alpha=0.9)

# Contour projection
ax.contour(X * 1e9, Y * 1e9, V, zdir='z', offset=-1.2, cmap='viridis', linewidths=0.8)

# Overlay wavefunction as translucent red-orange surface
wave_overlay = ax.plot_surface(X * 1e9, Y * 1e9, psi_multi * 2 - 1.2,
                               cmap='hot', alpha=0.5)

# Trap site markers
ax.scatter(trap_sites_nm[:, 0], trap_sites_nm[:, 1], trap_z, color='black', s=40, label='Trap sites')

# Formatting
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel('Potential / Wavefunction')
ax.set_title('Multiple Localized Wavefunctions in Periodic Potential')
ax.set_zlim(-1.2, 1.1)
fig.colorbar(surf, ax=ax, shrink=0.6, aspect=12, label='Potential (a.u.)')

plt.tight_layout()
plt.show()


In [ ]:
# Compute the probability density from the total wavefunction
prob_density = psi_multi**2  # normalized, so this is physical probability density

# Plot probability density surface alone
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot the probability density
density_surface = ax.plot_surface(X * 1e9, Y * 1e9, prob_density, cmap='plasma', edgecolor='none')

# Axes labels and title
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel(r'Probability Density $|\psi(x, y)|^2$')
ax.set_title('Probability Density of Multiple Localized States')
fig.colorbar(density_surface, ax=ax, shrink=0.6, aspect=12, label=r'$|\psi(x,y)|^2$')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate pairwise overlaps between the Gaussian wavefunctions centered at each trap site

num_sites = centers.shape[0]
overlap_matrix = np.zeros((num_sites, num_sites))

# Loop over all pairs and compute inner product (overlap integral)
for i in range(num_sites):
    for j in range(num_sites):
        x0_i, y0_i = centers[i]
        x0_j, y0_j = centers[j]
        psi_i = np.exp(-((X - x0_i)**2 + (Y - y0_i)**2) / (2 * sigma**2))
        psi_j = np.exp(-((X - x0_j)**2 + (Y - y0_j)**2) / (2 * sigma**2))
        norm_i = np.sqrt(np.sum(psi_i**2))
        norm_j = np.sqrt(np.sum(psi_j**2))
        overlap_matrix[i, j] = np.sum(psi_i * psi_j) / (norm_i * norm_j)

import pandas as pd
#import ace_tools as tools; tools.display_dataframe_to_user(
 #   name="Gaussian State Overlap Matrix",
  #  dataframe=pd.DataFrame(overlap_matrix, columns=[f"Site {i}" for i in range(num_sites)],
  #                         index=[f"Site {i}" for i in range(num_sites)])
#)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize the overlap matrix as a heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(overlap_matrix, annot=True, fmt=".2e", cmap='rocket', xticklabels=[f"Site {i}" for i in range(num_sites)],
            yticklabels=[f"Site {i}" for i in range(num_sites)])
plt.title("Gaussian State Overlap Matrix")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.animation as animation

# Choose two overlapping sites for tunneling animation
i, j = 0, 1  # Site 0 <-> Site 1
x0_i, y0_i = centers[i]
x0_j, y0_j = centers[j]

# Gaussian components
psi_i = np.exp(-((X - x0_i)**2 + (Y - y0_i)**2) / (2 * sigma**2))
psi_j = np.exp(-((X - x0_j)**2 + (Y - y0_j)**2) / (2 * sigma**2))
psi_i /= np.sqrt(np.sum(psi_i**2))
psi_j /= np.sqrt(np.sum(psi_j**2))

# Set up figure
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_zlim(0, np.max(psi_i**2 + psi_j**2) * 1.2)
ax.set_xlabel('x (nm)')
ax.set_ylabel('y (nm)')
ax.set_zlabel(r'$|\psi(x, y, t)|^2$')
ax.set_title('Coherent Tunneling Between Gaussian States')

# Initial wavefunction plot
wf_plot = [ax.plot_surface(X * 1e9, Y * 1e9, psi_i**2, cmap='plasma', edgecolor='none')]

# Time evolution parameters
frames = 60
t_vals = np.linspace(0, 2 * np.pi, frames)

# Time-dependent wavefunction: ψ(t) = cos(Ωt) ψ_i + sin(Ωt) ψ_j
def update(frame):
    t = t_vals[frame]
    psi_t = np.cos(t) * psi_i + np.sin(t) * psi_j
    prob_t = psi_t**2

    wf_plot[0].remove()
    wf_plot[0] = ax.plot_surface(X * 1e9, Y * 1e9, prob_t, cmap='plasma', edgecolor='none')
    return wf_plot[0],

ani = animation.FuncAnimation(fig, update, frames=frames, interval=100, blit=False)
plt.close(fig)

from IPython.display import HTML
HTML(ani.to_jshtml())


In [ ]:
!pip install scipy
from scipy import constants

In [ ]:
!pip install scipy
import scipy.integrate as simps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import e, k, h


# Simplified transmission function: resonant peaks at energy band centers
def transmission(E, V):
    peaks = np.arange(-1.5, 1.5, 0.01)  # band centers in eV
    width = 0.0005  # peak width in eV
    T = np.zeros_like(E)
    for peak in peaks:
        T += np.exp(-((E - peak)**2) / (2 * width**2))
    return T

# Fermi-Dirac distribution
def fermi(E, mu, T=0.01):
    beta = 1 / (k * T / e)  # eV^-1
    return 1 / (1 + np.exp(beta * (E - mu)))

# Energy grid
E = np.linspace(-1.5, 1.5, 1000)  # in eV

# Bias voltages and temperature
V_vals = np.linspace(-1.5, 1.5, 1000)
T_K = 0.01  # Temperature in Kelvin


# Recompute current using numpy.trapz instead
I_vals = []

for V in V_vals:
    mu_L = V / 2
    mu_R = -V / 2
    T_E = transmission(E, V)
    f_diff = fermi(E, mu_L, T=T_K) - fermi(E, mu_R, T=T_K)
    integrand = T_E * f_diff
    current = np.trapezoid(integrand, E)
    I_vals.append(current)

# Replot the I-V curve using trapezoidal integration
plt.figure(figsize=(12, 10))
plt.plot(V_vals, I_vals, color='darkgreen')
plt.xlabel('Voltage (V)')
plt.ylabel('Current (a.u.)')
plt.title('Simulated I-V Curve Using Trapezoidal Integration')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Re-import libraries after code execution environment reset
import numpy as np
import matplotlib.pyplot as plt

# Define voltage range similar to VF: 0 to 2.86 V
V = np.linspace(0, 2.86, 500)

# Generate a synthetic "butterfly" pattern in current (I in pA)
oscillations = np.sin(10 * np.pi * V)  # fast oscillation
envelope = 319 * (1 - np.abs(V - 1.43) / 1.43)  # symmetric envelope
I = envelope * oscillations  # modulated oscillations

# Plot using matplotlib
plt.figure(figsize=(8, 5))
plt.plot(V, I, color='lime', linewidth=1.5)
plt.xlabel('Voltage (V)')
plt.ylabel('Current (pA)')
plt.title('Synthetic Quantum Oscillation I-V Pattern')
plt.ylim(-350, 350)
plt.grid(True)
plt.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()


In [ ]:
# Re-import constants and fix missing ones
import numpy as np
import matplotlib.pyplot as plt

# Constants
hbar = 1.055e-34  # reduced Planck's constant (J·s)
e = 1.602e-19     # elementary charge (C)
m_eff = 0.19 * 9.11e-31  # effective electron mass in Si (kg)

# Parameters for toy model
k = np.linspace(-1e9, 1e9, 1000)  # wavevector in m^-1
alpha = 1e-11  # Rashba SOC strength (eV·m)
mu_values = np.linspace(-3e-3, 3e-3, 100)  # Sweep chemical potential (eV)
Delta = 1e-3  # superconducting gap (eV)

# Energy dispersion: Rashba SOC + proximity-induced superconductivity
def E_surface(k, mu, Delta):
    epsilon_k = (hbar * k)**2 / (2 * m_eff) / e - mu  # convert J to eV
    E = np.sqrt(epsilon_k**2 + (alpha * k)**2 + Delta**2)
    return E

# Plot
plt.figure(figsize=(10, 6))
for mu in mu_values:
    E = E_surface(k, mu, Delta)
    label = f"$\\mu$ = {mu*1e3:.1f} meV"
    plt.plot(k * 1e-9, E, label=label)

plt.title("Surface States Dispersion With Rashba SOC and Proximity SC")
plt.xlabel("Wavevector $k$ ($10^9$ m$^{-1}$)")
plt.ylabel("Energy $E$ (eV)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Set up figure and axis
fig, ax = plt.subplots(figsize=(10, 6))
line, = ax.plot([], [], lw=2)
title = ax.text(0.5, 1.05, '', transform=ax.transAxes, ha='center')

# Axis limits
ax.set_xlim(k.min() * 1e-9, k.max() * 1e-9)
ax.set_ylim(0, 0.006)
ax.set_xlabel("Wavevector $k$ ($10^9$ m$^{-1}$)")
ax.set_ylabel("Energy $E$ (eV)")
ax.grid(True)

# Sweep mu dynamically
mu_sweep = np.linspace(-5e-1, 5e-1, 100)

def init():
    line.set_data([], [])
    title.set_text("")
    return line, title

def animate(i):
    mu = mu_sweep[i]
    E = E_surface(k, mu, Delta)
    line.set_data(k * 1e-9, E)
    title.set_text(f"Surface State Dispersion, $\\mu$ = {mu*1e3:.2f} meV")
    return line, title

ani = animation.FuncAnimation(fig, animate, frames=len(mu_sweep),
                              init_func=init, blit=True, interval=100)

plt.close()  # Prevent duplicate static plot
ani
from IPython.display import HTML
HTML(ani.to_jshtml())